# Paper Latency and Call Analysis

This notebook builds paper-ready latency and call-count tables for `chatbs-base` and `biomni-base`. It reads the latest prediction run from each evaluation config, matching the loader rule in `evaluation_results.py` (newest experiment folder name with `RESULTS.jsonl`).

Notes on call definitions:
- **Latency** is `time_taken` from each raw prediction row, with `elapsed` as a fallback.
- **LLM calls** are counted from `token_usage.calls` where available; GRASP stores OpenAI usage inside assistant `messages`, so those usage-bearing assistant turns are counted instead.
- **SPARQL calls** are explicit query-bearing GRASP tool calls. For Ours, the result logs do not store literal SPARQL query text per retrieval, so the notebook uses the number of logged LWE retrieval/program steps (`intermediary_results`) as the available SPARQL/explorer-call proxy. Retrieval-only baselines have no SPARQL call log in these results and are counted as 0.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:  # pragma: no cover - used only outside the project env
    yaml = None

In [2]:
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "evaluation_results.py").exists():
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_DIR = REPO_ROOT / "paper_figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = [
    {"codename": "chatbs-base", "label": "ChatBS", "config": REPO_ROOT / "evaluations/chatbs-base/config.evaluation.yaml"},
    {"codename": "biomni-base", "label": "Biomni", "config": REPO_ROOT / "evaluations/biomni-base/config.evaluation.yaml"},
]

METHODS = [
    ("vectorsimilarity", "VSB"),
    ("grasp", "GRASP"),
    ("hipporag", "HippoRAG"),
    ("hypergraphrag", "HyperGRAG"),
    ("ours", "Ours"),
]

ROUND_DIGITS = 2

In [3]:
def _minimal_evaluation_config_parser(config_path: Path) -> dict[str, Any]:
    """Parse just the config keys this notebook needs if PyYAML is unavailable."""
    prediction_filename = "RESULTS.jsonl"
    prediction_dirs: dict[str, str] = {}
    in_prediction_dirs = False
    prediction_dirs_indent: int | None = None

    for raw_line in config_path.read_text().splitlines():
        line = raw_line.split("#", 1)[0].rstrip()
        if not line.strip():
            continue

        stripped = line.strip()
        indent = len(line) - len(line.lstrip(" "))

        if stripped.startswith("prediction_filename:"):
            value = stripped.split(":", 1)[1].strip().strip('"\'')
            prediction_filename = value or prediction_filename

        if stripped == "prediction_dirs:":
            in_prediction_dirs = True
            prediction_dirs_indent = indent
            continue

        if in_prediction_dirs:
            if prediction_dirs_indent is not None and indent <= prediction_dirs_indent:
                in_prediction_dirs = False
                prediction_dirs_indent = None
                continue
            if ":" in stripped:
                key, value = stripped.split(":", 1)
                value = value.strip().strip('"\'')
                if value:
                    prediction_dirs[key.strip()] = value

    return {
        "prediction_filename": prediction_filename,
        "prediction_dirs": prediction_dirs,
    }


def load_evaluation_config(config_path: Path) -> dict[str, Any]:
    if yaml is None:
        return _minimal_evaluation_config_parser(config_path)
    with config_path.open() as handle:
        return yaml.safe_load(handle)["evaluation"]


def resolve_repo_path(path: str | Path) -> Path:
    path = Path(path)
    return path if path.is_absolute() else REPO_ROOT / path


def latest_prediction_file(experiments_dir: str | Path, prediction_filename: str = "RESULTS.jsonl") -> Path:
    experiments_path = resolve_repo_path(experiments_dir)
    if not experiments_path.exists():
        raise FileNotFoundError(f"Prediction directory not found: {experiments_path}")

    experiment_dirs = sorted(
        (path for path in experiments_path.iterdir() if path.is_dir()),
        key=lambda path: path.name,
        reverse=True,
    )
    for experiment_dir in experiment_dirs:
        prediction_file = experiment_dir / prediction_filename
        if prediction_file.exists():
            return prediction_file

    raise FileNotFoundError(f"No {prediction_filename} found under {experiments_path}")


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    records = []
    with path.open() as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            if not isinstance(record, dict):
                raise TypeError(f"Expected object in {path}:{line_number}, got {type(record).__name__}")
            records.append(record)
    return records

In [4]:
def _parse_tool_args(args: Any) -> dict[str, Any]:
    if isinstance(args, dict):
        return args
    if isinstance(args, str):
        try:
            parsed = json.loads(args)
        except json.JSONDecodeError:
            return {}
        return parsed if isinstance(parsed, dict) else {}
    return {}


def count_llm_calls(record: dict[str, Any], method: str) -> float:
    calls = (record.get("token_usage") or {}).get("calls")
    if isinstance(calls, list):
        return float(len(calls))

    if method == "grasp":
        count = 0
        for message in record.get("messages") or []:
            content = message.get("content") if isinstance(message, dict) else None
            if isinstance(content, dict) and content.get("usage") is not None:
                count += 1
        return float(count)

    return np.nan


def count_sparql_calls(record: dict[str, Any], method: str) -> float:
    if method == "grasp":
        count = 0
        for message in record.get("messages") or []:
            content = message.get("content") if isinstance(message, dict) else None
            if not isinstance(content, dict):
                continue
            for tool_call in content.get("tool_calls") or []:
                args = _parse_tool_args(tool_call.get("args") or tool_call.get("arguments") or {})
                if args.get("query") is not None or args.get("sparql") is not None:
                    count += 1
        return float(count)

    if method == "ours":
        intermediary_results = record.get("intermediary_results")
        if isinstance(intermediary_results, list):
            return float(len(intermediary_results))
        return np.nan

    return 0.0


def prediction_latency(record: dict[str, Any]) -> float:
    value = record.get("time_taken", record.get("elapsed"))
    return float(value) if value is not None else np.nan

In [5]:
rows = []
run_rows = []

for dataset in DATASETS:
    settings = load_evaluation_config(dataset["config"])
    prediction_filename = settings.get("prediction_filename", "RESULTS.jsonl")
    prediction_dirs = dict(settings.get("prediction_dirs", {}))

    for method_code, method_label in METHODS:
        prediction_file = latest_prediction_file(prediction_dirs[method_code], prediction_filename)
        records = load_jsonl(prediction_file)

        for record in records:
            rows.append({
                "dataset": dataset["label"],
                "dataset_code": dataset["codename"],
                "method": method_label,
                "method_code": method_code,
                "question_id": record.get("id"),
                "latency_s": prediction_latency(record),
                "llm_calls": count_llm_calls(record, method_code),
                "sparql_calls": count_sparql_calls(record, method_code),
                "prediction_file": str(prediction_file.relative_to(REPO_ROOT)),
            })

        run_rows.append({
            "dataset": dataset["label"],
            "method": method_label,
            "n_examples": len(records),
            "prediction_file": str(prediction_file.relative_to(REPO_ROOT)),
        })

raw_df = pd.DataFrame(rows)
runs_df = pd.DataFrame(run_rows)

runs_df

,dataset,method,n_examples,prediction_file
0,ChatBS,VSB,103,evaluations/chatbs-base/explainer/vectorsimila...
1,ChatBS,GRASP,103,evaluations/chatbs-base/explainer/grasp/exp_20...
2,ChatBS,HippoRAG,103,evaluations/chatbs-base/explainer/hipporag/exp...
3,ChatBS,HyperGRAG,103,evaluations/chatbs-base/explainer/hypergraphra...
4,ChatBS,Ours,103,evaluations/chatbs-base/explainer/results/exp_...
5,Biomni,VSB,106,evaluations/biomni-base/explainer/vectorsimila...
6,Biomni,GRASP,106,evaluations/biomni-base/explainer/grasp/exp_20...
7,Biomni,HippoRAG,106,evaluations/biomni-base/explainer/hipporag/exp...
8,Biomni,HyperGRAG,106,evaluations/biomni-base/explainer/hypergraphra...
9,Biomni,Ours,106,evaluations/biomni-base/explainer/results/exp_...


In [6]:
dataset_summary = (
    raw_df.groupby(["method", "dataset"], observed=True)
    .agg(
        n_examples=("question_id", "count"),
        avg_latency_s=("latency_s", "mean"),
        avg_llm_calls=("llm_calls", "mean"),
        avg_sparql_calls=("sparql_calls", "mean"),
        llm_call_rows=("llm_calls", "count"),
        sparql_call_rows=("sparql_calls", "count"),
    )
    .reset_index()
)

method_order = {label: i for i, (_, label) in enumerate(METHODS)}
dataset_order = {dataset["label"]: i for i, dataset in enumerate(DATASETS)}

dataset_summary["method_order"] = dataset_summary["method"].map(method_order)
dataset_summary["dataset_order"] = dataset_summary["dataset"].map(dataset_order)
dataset_summary = dataset_summary.sort_values(["method_order", "dataset_order"]).drop(columns=["method_order", "dataset_order"])

dataset_summary

,method,dataset,n_examples,avg_latency_s,avg_llm_calls,avg_sparql_calls,llm_call_rows,sparql_call_rows
9,VSB,ChatBS,103,2.936029,1.000000,0.000000,103,103
8,VSB,Biomni,106,3.177072,1.000000,0.000000,106,106
1,GRASP,ChatBS,103,15.494195,7.786408,5.786408,103,103
0,GRASP,Biomni,106,10.565731,5.056604,1.632075,106,106
3,HippoRAG,ChatBS,103,4.592810,1.000000,0.000000,103,103
2,HippoRAG,Biomni,106,4.827487,1.000000,0.000000,106,106
5,HyperGRAG,ChatBS,103,12.667501,1.737864,0.000000,103,103
4,HyperGRAG,Biomni,106,14.111365,1.905660,0.000000,106,106
7,Ours,ChatBS,103,39.976108,6.951456,2.592233,103,103
6,Ours,Biomni,106,146.057010,9.320755,2.735849,106,106


In [7]:
compact_table = pd.DataFrame(index=[label for _, label in METHODS])
for metric, suffix in [
    ("avg_latency_s", "avg. latency (s)"),
    ("avg_llm_calls", "avg. LLM calls"),
    ("avg_sparql_calls", "avg. SPARQL calls"),
]:
    pivot = dataset_summary.pivot(index="method", columns="dataset", values=metric)
    compact_table[f"ChatBS {suffix}"] = pivot["ChatBS"]
    compact_table[f"Biomni {suffix}"] = pivot["Biomni"]

compact_table = compact_table.reset_index(names="Method")

compact_table_rounded = compact_table.copy()
for column in compact_table_rounded.columns.drop("Method"):
    compact_table_rounded[column] = compact_table_rounded[column].round(ROUND_DIGITS)

compact_table_rounded

,Method,ChatBS avg. latency (s),Biomni avg. latency (s),ChatBS avg. LLM calls,Biomni avg. LLM calls,ChatBS avg. SPARQL calls,Biomni avg. SPARQL calls
0,VSB,2.94,3.18,1.00,1.00,0.00,0.00
1,GRASP,15.49,10.57,7.79,5.06,5.79,1.63
2,HippoRAG,4.59,4.83,1.00,1.00,0.00,0.00
3,HyperGRAG,12.67,14.11,1.74,1.91,0.00,0.00
4,Ours,39.98,146.06,6.95,9.32,2.59,2.74


In [8]:
dataset_specific_table = pd.DataFrame(index=[label for _, label in METHODS])
for metric, suffix in [
    ("avg_latency_s", "avg. latency (s)"),
    ("avg_llm_calls", "avg. LLM calls"),
    ("avg_sparql_calls", "avg. SPARQL calls"),
]:
    pivot = dataset_summary.pivot(index="method", columns="dataset", values=metric)
    dataset_specific_table[f"ChatBS {suffix}"] = pivot["ChatBS"]
    dataset_specific_table[f"Biomni {suffix}"] = pivot["Biomni"]

dataset_specific_table = dataset_specific_table.reset_index(names="Method")
dataset_specific_table_rounded = dataset_specific_table.copy()
for column in dataset_specific_table_rounded.columns.drop("Method"):
    dataset_specific_table_rounded[column] = dataset_specific_table_rounded[column].round(ROUND_DIGITS)

dataset_specific_table_rounded

,Method,ChatBS avg. latency (s),Biomni avg. latency (s),ChatBS avg. LLM calls,Biomni avg. LLM calls,ChatBS avg. SPARQL calls,Biomni avg. SPARQL calls
0,VSB,2.94,3.18,1.00,1.00,0.00,0.00
1,GRASP,15.49,10.57,7.79,5.06,5.79,1.63
2,HippoRAG,4.59,4.83,1.00,1.00,0.00,0.00
3,HyperGRAG,12.67,14.11,1.74,1.91,0.00,0.00
4,Ours,39.98,146.06,6.95,9.32,2.59,2.74


In [9]:
compact_csv = OUTPUT_DIR / "latency_call_summary.csv"
compact_tex = OUTPUT_DIR / "latency_call_summary.tex"
dataset_csv = OUTPUT_DIR / "latency_call_summary_by_dataset.csv"
dataset_tex = OUTPUT_DIR / "latency_call_summary_by_dataset.tex"
run_csv = OUTPUT_DIR / "latency_call_source_runs.csv"

compact_table_rounded.to_csv(compact_csv, index=False)
dataset_specific_table_rounded.to_csv(dataset_csv, index=False)
runs_df.to_csv(run_csv, index=False)

def polish_latex_table(latex: str, size: str = r"\small") -> str:
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")
    latex = latex.replace("\\begin{tabular}", f"{size}\n" + "\\begin{tabular}")
    return latex

compact_latex = compact_table_rounded.to_latex(
    index=False,
    escape=False,
    column_format="lrrrrrr",
    float_format=lambda value: f"{value:.2f}",
    caption=(
        "Dataset-specific average latency and call counts for each baseline and Ours. "
        "SPARQL calls are explicit query-bearing tool calls for GRASP; for Ours, the logged LWE "
        "retrieval/program steps are used as the available SPARQL/explorer-call proxy."
    ),
    label="tab:latency-call-summary",
)
compact_tex.write_text(polish_latex_table(compact_latex, size=r"\scriptsize"))

dataset_latex = dataset_specific_table_rounded.to_latex(
    index=False,
    escape=False,
    column_format="lrrrrrr",
    float_format=lambda value: f"{value:.2f}",
    caption=(
        "Dataset-specific latency and call counts for each baseline and Ours. "
        "SPARQL calls follow the same logging definition as Table~\\ref{tab:latency-call-summary}."
    ),
    label="tab:latency-call-summary-by-dataset",
)
dataset_tex.write_text(polish_latex_table(dataset_latex, size=r"\scriptsize"))

print(f"Saved {compact_csv.relative_to(REPO_ROOT)}")
print(f"Saved {compact_tex.relative_to(REPO_ROOT)}")
print(f"Saved {dataset_csv.relative_to(REPO_ROOT)}")
print(f"Saved {dataset_tex.relative_to(REPO_ROOT)}")
print(f"Saved {run_csv.relative_to(REPO_ROOT)}")

Saved paper_figures/latency_call_summary.csv
Saved paper_figures/latency_call_summary.tex
Saved paper_figures/latency_call_summary_by_dataset.csv
Saved paper_figures/latency_call_summary_by_dataset.tex
Saved paper_figures/latency_call_source_runs.csv


In [10]:
# Quick availability checks: these should equal n_examples when a metric is available for the method/dataset.
availability = dataset_summary[[
    "method",
    "dataset",
    "n_examples",
    "llm_call_rows",
    "sparql_call_rows",
]].copy()
availability

,method,dataset,n_examples,llm_call_rows,sparql_call_rows
9,VSB,ChatBS,103,103,103
8,VSB,Biomni,106,106,106
1,GRASP,ChatBS,103,103,103
0,GRASP,Biomni,106,106,106
3,HippoRAG,ChatBS,103,103,103
2,HippoRAG,Biomni,106,106,106
5,HyperGRAG,ChatBS,103,103,103
4,HyperGRAG,Biomni,106,106,106
7,Ours,ChatBS,103,103,103
6,Ours,Biomni,106,106,106
